In [13]:
# MASTER SETUP CELL — run this after every reconnect
import os, cv2, torch, numpy as np, pandas as pd
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm, time

# 1. Manifest
os.system("wget -q https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv")
manifest = pd.read_csv('split_manifest.csv')

# 2. FFT function
def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    return (norm_3ch - mean) / std

# 3. Dataset class
class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.samples = []
        split_df = manifest[manifest['split'] == split]
        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'
            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder): continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))
        print(f"[{split}] {compression_levels}: {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)
        return rgb_tensor, fft_tensor, torch.tensor(label, dtype=torch.float32)

# 4. Model class
class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        return self.classifier(self.dropout(fused))

# 5. Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 6. Datasets (skip train/val — only need test for evaluation)
test_dataset_original = CelebDFDataset(manifest, 'test', ['original'])

print("Setup complete — ready for evaluation")

Device: cuda
[test] ['original']: 7499 samples
Setup complete — ready for evaluation


In [3]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-24 08:28:10--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv’

split_manifest.csv  100%[===================>]  61.62K  --.-KB/s    in 0.01s   

2026-07-24 08:28:10 (5.06 MB/s) - ‘split_manifest.csv’ saved [63103/63103]



In [4]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-24 08:28:10--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv.1’

split_manifest.csv. 100%[===================>]  61.62K  --.-KB/s    in 0.01s   

2026-07-24 08:28:10 (5.23 MB/s) - ‘split_manifest.csv.1’ saved [63103/63103]



In [5]:
import pandas as pd
manifest = pd.read_csv('split_manifest.csv')
print(manifest.shape)
manifest.head()

(1203, 4)


,video_path,label,source,split
0,YouTube-real/00000.mp4,1,youtube-real,train
1,Celeb-real/id4_0001.mp4,1,celeb-real,train
2,Celeb-real/id10_0007.mp4,1,celeb-real,train
3,YouTube-real/00079.mp4,1,youtube-real,train
4,Celeb-real/id11_0004.mp4,1,celeb-real,train


In [6]:
print(manifest['label'].unique())
print(manifest['split'].unique())
print(manifest['source'].unique())
print(manifest['label'].value_counts())
print(manifest['split'].value_counts())

[1 0]
['train' 'val' 'test']
['youtube-real' 'celeb-real' 'celeb-synthesis']
label
0    795
1    408
Name: count, dtype: int64
split
train    841
test     182
val      180
Name: count, dtype: int64


In [7]:
print(manifest.groupby(['source', 'label']).size())

source           label
celeb-real       1        158
celeb-synthesis  0        795
youtube-real     1        250
dtype: int64


In [8]:
!mkdir -p ~/.kaggle
!echo KGAT_ea2cc5f42eab7752c7e422757de57a92 > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

In [9]:
!pip install -q kaggle
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces

Dataset URL: https://www.kaggle.com/datasets/niranjana1310/celeb-df-v2-faces
License(s): CC0-1.0
100% 910M/910M [00:11<00:00, 84.5MB/s]



In [10]:
!unzip -q celeb-df-v2-faces.zip -d faces_data
!ls faces_data

test  train  val


In [11]:
!ls faces_data/train

fake  real


In [12]:
!ls faces_data/train/real
!echo "---"
!ls faces_data/train/fake

crf23  crf28  crf35  original
---
crf23  crf28  crf35  original


In [13]:
!ls faces_data/train/real/original | head -10

00000_frame00000.jpg
00000_frame00010.jpg
00000_frame00020.jpg
00000_frame00030.jpg
00000_frame00040.jpg
00000_frame00050.jpg
00000_frame00060.jpg
00000_frame00070.jpg
00000_frame00080.jpg
00000_frame00090.jpg


In [14]:
!ls faces_data/train/real/original | grep id4 | head -5

id4_0001_frame00000.jpg
id4_0001_frame00010.jpg
id4_0001_frame00020.jpg
id4_0001_frame00030.jpg
id4_0001_frame00040.jpg


In [15]:
def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fft_input = (norm_3ch - mean) / std
    return fft_input

In [16]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.root_dir = root_dir
        self.compression_levels = compression_levels

        # Filter manifest to just this split (train/val/test)
        split_df = manifest[manifest['split'] == split]

        # Build a list of (image_path, label) for every matching frame
        self.samples = []

        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'

            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder):
                    continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))

        print(f"[{split}] compression={compression_levels}: {len(self.samples)} image samples found")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))  # safety, should already be 224x224

        # RGB tensor (ImageNet normalized)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)

        # FFT tensor
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)

        label_tensor = torch.tensor(label, dtype=torch.float32)

        return rgb_tensor, fft_tensor, label_tensor

In [17]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])

[train] compression=['original']: 34644 image samples found


In [18]:
!wget https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv

--2026-07-24 08:29:30--  https://raw.githubusercontent.com/1310Niranjana/mini-project/preprocessing/metadata/split_manifest.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63103 (62K) [text/plain]
Saving to: ‘split_manifest.csv.2’

split_manifest.csv. 100%[===================>]  61.62K  --.-KB/s    in 0.01s   

2026-07-24 08:29:30 (5.07 MB/s) - ‘split_manifest.csv.2’ saved [63103/63103]



In [19]:
import pandas as pd
manifest = pd.read_csv('split_manifest.csv')
print(manifest.shape)
manifest.head()

(1203, 4)


,video_path,label,source,split
0,YouTube-real/00000.mp4,1,youtube-real,train
1,Celeb-real/id4_0001.mp4,1,celeb-real,train
2,Celeb-real/id10_0007.mp4,1,celeb-real,train
3,YouTube-real/00079.mp4,1,youtube-real,train
4,Celeb-real/id11_0004.mp4,1,celeb-real,train


In [20]:
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[val] compression=['original']: 7198 image samples found


In [21]:
rgb_sample, fft_sample, label_sample = train_dataset[0]
print(rgb_sample.shape, fft_sample.shape, label_sample)

torch.Size([3, 224, 224]) torch.Size([3, 224, 224]) tensor(1.)


In [22]:
!pip install -q timm

In [23]:
import torch.nn as nn
import timm

class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        dropped = self.dropout(fused)
        logit = self.classifier(dropped)
        return logit

In [24]:
print(manifest.shape)

(1203, 4)


In [25]:
def fft_preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = np.fft.fft2(gray)
    f_shifted = np.fft.fftshift(f)
    magnitude = np.abs(f_shifted)
    log_magnitude = np.log1p(magnitude)
    norm = (log_magnitude - log_magnitude.min()) / (log_magnitude.max() - log_magnitude.min())
    norm_3ch = np.stack([norm, norm, norm], axis=-1)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    fft_input = (norm_3ch - mean) / std
    return fft_input

In [26]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class CelebDFDataset(Dataset):
    def __init__(self, manifest, split, compression_levels, root_dir='faces_data'):
        self.root_dir = root_dir
        self.compression_levels = compression_levels
        split_df = manifest[manifest['split'] == split]
        self.samples = []
        for _, row in split_df.iterrows():
            video_name = os.path.splitext(os.path.basename(row['video_path']))[0]
            label = row['label']
            class_folder = 'real' if label == 1 else 'fake'
            for comp in compression_levels:
                folder = os.path.join(root_dir, split, class_folder, comp)
                if not os.path.isdir(folder):
                    continue
                for fname in os.listdir(folder):
                    if fname.startswith(video_name + '_frame'):
                        self.samples.append((os.path.join(folder, fname), label))
        print(f"[{split}] compression={compression_levels}: {len(self.samples)} image samples found")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        rgb_norm = (img / 255.0 - mean) / std
        rgb_tensor = torch.tensor(rgb_norm, dtype=torch.float32).permute(2, 0, 1)
        fft_img = fft_preprocess(img)
        fft_tensor = torch.tensor(fft_img, dtype=torch.float32).permute(2, 0, 1)
        label_tensor = torch.tensor(label, dtype=torch.float32)
        return rgb_tensor, fft_tensor, label_tensor

In [27]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[train] compression=['original']: 34644 image samples found
[val] compression=['original']: 7198 image samples found


In [28]:
import timm
print(timm.__version__)

1.0.28


In [29]:
import torch.nn as nn
import timm

class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.fft_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(2560, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        fft_feat = self.fft_branch(fft_input)
        fused = torch.cat([rgb_feat, fft_feat], dim=1)
        dropped = self.dropout(fused)
        logit = self.classifier(dropped)
        return logit

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [31]:
print(manifest.shape)

(1203, 4)


In [32]:
print(len(train_dataset))
print(len(val_dataset))

34644
7198


In [33]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [34]:
rgb_batch, fft_batch, label_batch = next(iter(train_loader))
print(rgb_batch.shape, fft_batch.shape, label_batch.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32, 3, 224, 224]) torch.Size([32])


In [35]:
model = DualStreamModel().to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

In [36]:
import torch.optim as optim

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [37]:
model.train()

for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
    if batch_idx >= 3:
        break

    rgb_batch = rgb_batch.to(device)
    fft_batch = fft_batch.to(device)
    label_batch = label_batch.to(device).unsqueeze(1)

    optimizer.zero_grad()
    logits = model(rgb_batch, fft_batch)
    loss = criterion(logits, label_batch)
    loss.backward()
    optimizer.step()

    print(f"Batch {batch_idx} - Loss: {loss.item():.4f}")

print("Test run complete - no errors!")

Batch 0 - Loss: 0.6800
Batch 1 - Loss: 0.7176
Batch 2 - Loss: 0.6849
Test run complete - no errors!


In [38]:
import time

num_epochs = 10
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} ({elapsed/60:.1f} min) — "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    ckpt_name = f'model_baseline_epoch{epoch+1}.pth'
    torch.save(model.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"Checkpoint saved: {ckpt_name}")

  Batch 0/1083 - Loss: 0.6887
  Batch 100/1083 - Loss: 0.1508
  Batch 200/1083 - Loss: 0.1964
  Batch 300/1083 - Loss: 0.0977
  Batch 400/1083 - Loss: 0.0215
  Batch 500/1083 - Loss: 0.1101
  Batch 600/1083 - Loss: 0.0272
  Batch 700/1083 - Loss: 0.0966
  Batch 800/1083 - Loss: 0.0120
  Batch 900/1083 - Loss: 0.1082
  Batch 1000/1083 - Loss: 0.0167

Epoch 1/10 (7.0 min) — Train Loss: 0.0976, Train Acc: 0.9622 | Val Loss: 0.1861, Val Acc: 0.9351


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checkpoint saved: model_baseline_epoch1.pth
  Batch 0/1083 - Loss: 0.0060
  Batch 100/1083 - Loss: 0.0247


KeyboardInterrupt: 

In [ ]:
print(num_epochs)

In [ ]:
model = DualStreamModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("Model reset - ready for full training")

In [ ]:
print(len(train_dataset), len(val_dataset), device)

In [ ]:
class RGBOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_branch = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(1280, 1)

    def forward(self, rgb_input, fft_input):
        rgb_feat = self.rgb_branch(rgb_input)
        dropped = self.dropout(rgb_feat)
        logit = self.classifier(dropped)
        return logit

In [ ]:
model_rgb = RGBOnlyModel().to(device)
optimizer_rgb = optim.Adam(model_rgb.parameters(), lr=0.0001)
print("RGB-only model ready")

In [ ]:
import time

num_epochs = 10
history_rgb = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model_rgb.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer_rgb.zero_grad()
        logits = model_rgb(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer_rgb.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model_rgb.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model_rgb(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    history_rgb['train_loss'].append(train_loss)
    history_rgb['train_acc'].append(train_acc)
    history_rgb['val_loss'].append(val_loss)
    history_rgb['val_acc'].append(val_acc)

    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} ({elapsed/60:.1f} min) — "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    ckpt_name = f'model_rgb_only_epoch{epoch+1}.pth'
    torch.save(model_rgb.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"Checkpoint saved: {ckpt_name}\n")

In [ ]:
print(manifest.shape)

In [ ]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

In [ ]:
!ls faces_data/train/real/original | head -5

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!echo KGAT_ea2cc5f42eab7752c7e422757de57a92 > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

In [41]:
train_dataset = CelebDFDataset(manifest, split='train', compression_levels=['original'])
val_dataset = CelebDFDataset(manifest, split='val', compression_levels=['original'])

[train] compression=['original']: 34644 image samples found
[val] compression=['original']: 7198 image samples found


In [47]:
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

cuda


In [48]:
import torch.optim as optim

model = DualStreamModel().to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
print("Ready")

Ready


In [43]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
test_original = CelebDFDataset(manifest, split='test', compression_levels=['original'])
test_crf23 = CelebDFDataset(manifest, split='test', compression_levels=['crf23'])
test_crf28 = CelebDFDataset(manifest, split='test', compression_levels=['crf28'])
test_crf35 = CelebDFDataset(manifest, split='test', compression_levels=['crf35'])

In [ ]:
baseline_model = DualStreamModel().to(device)
baseline_model.load_state_dict(torch.load('model_baseline_epoch7.pth', map_location=device))
baseline_model.eval()
print("Baseline model loaded")

In [ ]:
def evaluate_model(model, dataset, compression_name):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
    correct = 0
    total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)

            logits = model(rgb_batch, fft_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == label_batch).sum().item()
            total += label_batch.size(0)

    accuracy = correct / total
    print(f"{compression_name}: {accuracy:.4f} ({accuracy*100:.2f}%)")
    return accuracy

In [ ]:
print("=== BASELINE MODEL (epoch7) on TEST SET ===")
acc_original = evaluate_model(baseline_model, test_original, "Original (clean)")
acc_crf23 = evaluate_model(baseline_model, test_crf23, "CRF23 (mild)")
acc_crf28 = evaluate_model(baseline_model, test_crf28, "CRF28 (medium)")
acc_crf35 = evaluate_model(baseline_model, test_crf35, "CRF35 (heavy)")

print(f"\n=== SUMMARY ===")
print(f"Original: {acc_original*100:.2f}%")
print(f"CRF23:    {acc_crf23*100:.2f}%")
print(f"CRF28:    {acc_crf28*100:.2f}%")
print(f"CRF35:    {acc_crf35*100:.2f}%")

In [ ]:
train_dataset_compressed = CelebDFDataset(
    manifest,
    split='train',
    compression_levels=['original', 'crf23', 'crf28', 'crf35']
)

val_dataset_compressed = CelebDFDataset(
    manifest,
    split='val',
    compression_levels=['original', 'crf23', 'crf28', 'crf35']
)

In [40]:
train_loader_compressed = DataLoader(train_dataset_compressed, batch_size=32, shuffle=True, num_workers=2)
val_loader_compressed = DataLoader(val_dataset_compressed, batch_size=32, shuffle=False, num_workers=2)

# Fresh model for compression-aware training
model_ca = DualStreamModel().to(device)
optimizer_ca = optim.Adam(model_ca.parameters(), lr=0.0001)
print(f"Compression-aware training ready")
print(f"Train batches per epoch: {len(train_loader_compressed)}")

NameError: name 'train_dataset_compressed' is not defined

In [ ]:
import time

num_epochs = 5  # Session 1: epochs 1-5
history_ca = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    start_time = time.time()

    model_ca.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (rgb_batch, fft_batch, label_batch) in enumerate(train_loader_compressed):
        rgb_batch = rgb_batch.to(device)
        fft_batch = fft_batch.to(device)
        label_batch = label_batch.to(device).unsqueeze(1)

        optimizer_ca.zero_grad()
        logits = model_ca(rgb_batch, fft_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer_ca.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == label_batch).sum().item()
        total += label_batch.size(0)

        if batch_idx % 500 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader_compressed)} | Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader_compressed)
    train_acc = correct / total

    model_ca.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in val_loader_compressed:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model_ca(rgb_batch, fft_batch)
            loss = criterion(logits, label_batch)
            val_running_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = val_running_loss / len(val_loader_compressed)
    val_acc = val_correct / val_total

    history_ca['train_loss'].append(train_loss)
    history_ca['train_acc'].append(train_acc)
    history_ca['val_loss'].append(val_loss)
    history_ca['val_acc'].append(val_acc)

    elapsed = time.time() - start_time

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1}/10 COMPLETE")
    print(f"Time: {elapsed/60:.1f} minutes")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
    print(f"{'='*60}\n")

    ckpt_name = f'model_compression_aware_epoch{epoch+1}.pth'
    torch.save(model_ca.state_dict(), ckpt_name)
    from google.colab import files
    files.download(ckpt_name)
    print(f"✓ Checkpoint saved and downloading: {ckpt_name}\n")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
model_baseline = DualStreamModel().to(device)
model_baseline.load_state_dict(torch.load('model_baseline_epoch7.pth', map_location=device))
model_baseline.eval()

model_ca = DualStreamModel().to(device)
model_ca.load_state_dict(torch.load('model_compression_aware_epoch1.pth', map_location=device))
model_ca.eval()
print("Both models loaded")

In [14]:
import gdown

# Download baseline model (epoch 7)
gdown.download('https://drive.google.com/uc?id=14apuWcEfqwdHFZFZYXBXfQZNfikc4G7n',
               'model_baseline_epoch7.pth', quiet=False)

# Download compression-aware model (epoch 1)
gdown.download('https://drive.google.com/uc?id=1PB2rrwpO7JYBiXoYBSbtYBxMCAPPvQpW',
               'model_compression_aware_epoch1.pth', quiet=False)

print("Both models downloaded")

Downloading...
From (original): https://drive.google.com/uc?id=14apuWcEfqwdHFZFZYXBXfQZNfikc4G7n
From (redirected): https://drive.google.com/uc?id=14apuWcEfqwdHFZFZYXBXfQZNfikc4G7n&confirm=t&uuid=308a994f-61f3-458f-9067-20cc3ebc1cce
To: /content/model_baseline_epoch7.pth
100%|██████████| 32.7M/32.7M [00:00<00:00, 102MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1PB2rrwpO7JYBiXoYBSbtYBxMCAPPvQpW
From (redirected): https://drive.google.com/uc?id=1PB2rrwpO7JYBiXoYBSbtYBxMCAPPvQpW&confirm=t&uuid=f2112fdc-d48f-488e-8ab7-05092b3ef8e7
To: /content/model_compression_aware_epoch1.pth
100%|██████████| 32.7M/32.7M [00:00<00:00, 73.5MB/s]

Both models downloaded


In [4]:
model_baseline = DualStreamModel().to(device)
model_baseline.load_state_dict(torch.load('model_baseline_epoch7.pth', map_location=device))
model_baseline.eval()

model_ca = DualStreamModel().to(device)
model_ca.load_state_dict(torch.load('model_compression_aware_epoch1.pth', map_location=device))
model_ca.eval()

print("Both models loaded successfully")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Both models loaded successfully


In [5]:
def evaluate_model(model, compression_level):
    test_dataset = CelebDFDataset(manifest, split='test', compression_levels=[compression_level])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in test_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model(rgb_batch, fft_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == label_batch).sum().item()
            total += label_batch.size(0)
    acc = correct / total
    print(f"[{compression_level}] Test Accuracy: {acc*100:.2f}%")
    return acc

print("=== BASELINE MODEL ===")
for comp in ['original', 'crf23', 'crf28', 'crf35']:
    evaluate_model(model_baseline, comp)

print("\n=== COMPRESSION-AWARE MODEL ===")
for comp in ['original', 'crf23', 'crf28', 'crf35']:
    evaluate_model(model_ca, comp)

=== BASELINE MODEL ===
[test] ['original']: 0 samples


ZeroDivisionError: division by zero

In [6]:
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -q -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open celeb-df-v2-faces.zip, celeb-df-v2-faces.zip.zip or celeb-df-v2-faces.zip.ZIP.
ls: cannot access 'faces_data': No such file or directory


In [7]:
import os
os.environ['KAGGLE_TOKEN'] = 'KGAT_ea2cc5f42eab7752c7e422757de57a92'

!mkdir -p ~/.kaggle
!echo '{"username":"kaganya","key":"KGAT_ea2cc5f42eab7752c7e422757de57a92"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -q -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open celeb-df-v2-faces.zip, celeb-df-v2-faces.zip.zip or celeb-df-v2-faces.zip.ZIP.
ls: cannot access 'faces_data': No such file or directory


In [8]:
!mkdir -p ~/.kaggle
!echo '{"username":"kaganya","key":"KGAT_3077acd90fb3d7072044742741958768"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -q -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
unzip:  cannot find or open celeb-df-v2-faces.zip, celeb-df-v2-faces.zip.zip or celeb-df-v2-faces.zip.ZIP.
ls: cannot access 'faces_data': No such file or directory


In [9]:
!curl -I "https://www.kaggle.com/datasets/niranjana1310/celeb-df-v2-faces"

HTTP/2 404 
content-length: 134
content-type: text/html; charset=UTF-8
date: Fri, 24 Jul 2026 09:16:39 GMT
alt-svc: h3=":443"; ma=2592000,h3-29=":443"; ma=2592000



In [10]:
!ls /content/faces_data 2>/dev/null || echo "NOT FOUND"

NOT FOUND


In [15]:
!kaggle datasets download -d niranjana1310/celeb-df-v2-faces
!unzip -q -o celeb-df-v2-faces.zip -d faces_data
!ls faces_data

Dataset URL: https://www.kaggle.com/datasets/niranjana1310/celeb-df-v2-faces
License(s): CC0-1.0
celeb-df-v2-faces.zip: Skipping, found more recently modified local copy (use --force to force download)
test  train  val


In [12]:
def evaluate_model(model, compression_level):
    test_dataset = CelebDFDataset(manifest, split='test', compression_levels=[compression_level])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in test_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model(rgb_batch, fft_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == label_batch).sum().item()
            total += label_batch.size(0)
    acc = correct / total
    print(f"[{compression_level}] Test Accuracy: {acc*100:.2f}%")
    return acc

print("=== BASELINE MODEL ===")
for comp in ['original', 'crf23', 'crf28', 'crf35']:
    evaluate_model(model_baseline, comp)

print("\n=== COMPRESSION-AWARE MODEL ===")
for comp in ['original', 'crf23', 'crf28', 'crf35']:
    evaluate_model(model_ca, comp)

=== BASELINE MODEL ===
[test] ['original']: 7499 samples
[original] Test Accuracy: 95.43%
[test] ['crf23']: 7498 samples
[crf23] Test Accuracy: 93.60%
[test] ['crf28']: 7498 samples
[crf28] Test Accuracy: 90.89%
[test] ['crf35']: 7499 samples
[crf35] Test Accuracy: 68.94%

=== COMPRESSION-AWARE MODEL ===
[test] ['original']: 7499 samples
[original] Test Accuracy: 93.00%
[test] ['crf23']: 7498 samples
[crf23] Test Accuracy: 92.58%
[test] ['crf28']: 7498 samples
[crf28] Test Accuracy: 90.92%
[test] ['crf35']: 7499 samples
[crf35] Test Accuracy: 88.11%


In [16]:
def evaluate_model(model, compression_level):
    test_dataset = CelebDFDataset(manifest, split='test', compression_levels=[compression_level])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for rgb_batch, fft_batch, label_batch in test_loader:
            rgb_batch = rgb_batch.to(device)
            fft_batch = fft_batch.to(device)
            label_batch = label_batch.to(device).unsqueeze(1)
            logits = model(rgb_batch, fft_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == label_batch).sum().item()
            total += label_batch.size(0)
    acc = correct / total
    print(f"[{compression_level}] Test Accuracy: {acc*100:.2f}%")
    return acc

print("=== BASELINE CRF35 ONLY ===")
evaluate_model(model_baseline, 'crf35')

print("\n=== COMPRESSION-AWARE MODEL ===")
for comp in ['original', 'crf23', 'crf28', 'crf35']:
    evaluate_model(model_ca, comp)

=== BASELINE CRF35 ONLY ===
[test] ['crf35']: 7499 samples
[crf35] Test Accuracy: 68.94%

=== COMPRESSION-AWARE MODEL ===
[test] ['original']: 7499 samples
[original] Test Accuracy: 93.00%
[test] ['crf23']: 7498 samples
[crf23] Test Accuracy: 92.58%
[test] ['crf28']: 7498 samples
[crf28] Test Accuracy: 90.92%
[test] ['crf35']: 7499 samples
[crf35] Test Accuracy: 88.11%


In [17]:
import torch
import timm
import cv2
import numpy as np
import pandas as pd

print(f"torch=={torch.__version__}")
print(f"timm=={timm.__version__}")
print(f"opencv-python=={cv2.__version__}")
print(f"numpy=={np.__version__}")
print(f"pandas=={pd.__version__}")

torch==2.11.0+cu128
timm==1.0.28
opencv-python==4.13.0
numpy==2.0.2
pandas==2.2.2


In [18]:
requirements = """torch==2.11.0+cu128
torchvision==0.22.0
timm==1.0.28
opencv-python==4.13.0
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
Pillow==11.2.1
matplotlib==3.10.0
streamlit==1.45.1
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt created")

from google.colab import files
files.download('requirements.txt')

requirements.txt created


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>